# 1. Quickstart: CivicHonors Knowledge Reduction

This notebook demonstrates the production `knowledgereduce` pipeline replacing the legacy Colab notebooks (`legacy/notebooks/`). It ingests multi-source CivicHonors data, extracts grounded `Entity`, `Claim`, and `Relation` records, resolves duplicates & preserves disputes, and persists an immutable DuckDB + Parquet layer.

In [ ]:
import tempfile
from pathlib import Path

from knowledgereduce import (
    ConflictStrategy,
    DuckDBParquetStore,
    IntegrityValidator,
    LocalFileSourceConfig,
    PipelineConfig,
    PipelineRunner,
)

In [ ]:
# Locate repository fixtures cleanly whether run from repo root or notebooks/
cwd = Path.cwd()
repo_root = cwd if (cwd / "tests" / "fixtures").is_dir() else cwd.parent
fixtures = repo_root / "tests" / "fixtures"

with tempfile.TemporaryDirectory() as tmp:
    tmp_path = Path(tmp)
    prose = tmp_path / "civichonors_prose.txt"
    prose.write_text(
        "Civic Honors was published in 2006 in Denver, Colorado.\n"
        "Dr. Nels Lindahl studied at University of Kansas under Dr. H. George Frederickson.\n"
        "Civic Honors was published in 2020 in Denver, Colorado.\n",
        encoding="utf-8",
    )

    config = PipelineConfig(
        layer_id="civichonors-nb-1",
        layer_name="Civic Honors Quickstart",
        store_dir=str(tmp_path / "store"),
        checkpoint_dir=str(tmp_path / "checkpoints"),
        local_files=[
            LocalFileSourceConfig(path=str(fixtures / "civichonors_facts.csv"), source_id="csv", trust_weight=0.95),
            LocalFileSourceConfig(path=str(fixtures / "civichonors_notes.json"), source_id="json", trust_weight=0.90),
            LocalFileSourceConfig(path=str(prose), source_id="prose", trust_weight=0.65),
        ],
        conflict_strategy=ConflictStrategy.PRESERVE_DISPUTE,
        min_sentence_length=20,
    )

    layer_data = PipelineRunner().run(config)
    store = DuckDBParquetStore(tmp_path / "store")
    report = IntegrityValidator().validate(store.load_layer("civichonors-nb-1"), strict=True)
    print("Valid:", report.valid, "| Metrics:", report.metrics)